In [1]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.ensemble import IsolationForest

from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)


# =========================================================
# Load datasets
# =========================================================

BASELINE_RAW_WINDOWS_PATH = (
    "../data/processed/baseline_raw_windows.parquet"
)

WINDOW_FEATURES_PATH = (
    "../data/processed/window_features.parquet"
)

TEMPORAL_FEATURES_PATH = (
    "../data/processed/temporal_features.parquet"
)

ML_FEATURES_PATH = (
    "../data/processed/ml_features.parquet"
)


baseline_raw_windows = pd.read_parquet(
    BASELINE_RAW_WINDOWS_PATH
)

window_features = pd.read_parquet(
    WINDOW_FEATURES_PATH
)

temporal_features = pd.read_parquet(
    TEMPORAL_FEATURES_PATH
)

ml_features = pd.read_parquet(
    ML_FEATURES_PATH
)


# =========================================================
# Create observable anomaly target
# =========================================================

def add_observable_anomaly_target(df):
    """
    Original anomaly_label:

        0 = normal
        1 = unobservable fault
        2 = observable anomaly
        3 = observable aftereffect

    New target:

        0 = normal
        1 = observable anomaly
        NaN = unobservable fault
    """

    df = df.copy()

    df["observable_anomaly"] = np.where(
        df["anomaly_label"].isin([2, 3]),
        1,
        np.where(
            df["anomaly_label"] == 0,
            0,
            np.nan
        )
    )

    return df


# baseline_raw_windows already contains the target.
# The other feature datasets receive it here.

window_features = add_observable_anomaly_target(
    window_features
)

temporal_features = add_observable_anomaly_target(
    temporal_features
)

ml_features = add_observable_anomaly_target(
    ml_features
)


# =========================================================
# Metadata columns
# =========================================================

metadata_cols = [
    "identifier",
    "window",
    "window_start",
    "window_end",
    "phase",
    "batch",
    "operating_point",
    "experiment_type",
    "experiment",
    "anomaly_label",
]


# =========================================================
# MODEL 1
# Baseline → baseline_raw_windows
#
# Raw 60-second signal values
# =========================================================

baseline_data = baseline_raw_windows.copy()


# ---------------------------------------------------------
# Identify raw signal features
# ---------------------------------------------------------

baseline_features = [
    c
    for c in baseline_data.columns
    if c not in metadata_cols
    and c != "observable_anomaly"
]


print(
    f"Model 1: {len(baseline_features)} raw signal features"
)


# =========================================================
# MODEL 2
# Raw Window → window_features
#
# Statistical/process features
# =========================================================

raw_window_data = window_features.copy()

raw_window_features = [
    c
    for c in raw_window_data.columns
    if c not in metadata_cols
    and c != "observable_anomaly"
]


# =========================================================
# MODEL 3
# Temporal → temporal_features
#
# Previous-window + delta features
# =========================================================

temporal_data = temporal_features.copy()

temporal_feature_cols = [
    c
    for c in temporal_data.columns
    if c not in metadata_cols
    and c != "observable_anomaly"
]


# =========================================================
# MODEL 4
# Temporal + Frequency → ml_features
#
# Temporal + frequency-domain features
# =========================================================

temporal_frequency_data = ml_features.copy()

temporal_frequency_features = [
    c
    for c in temporal_frequency_data.columns
    if c not in metadata_cols
    and c != "observable_anomaly"
]


# =========================================================
# Remove frequency features with excessive missingness
# =========================================================

MAX_MISSING_FRACTION = 0.95

missing_fraction = (
    temporal_frequency_data[
        temporal_frequency_features
    ]
    .isna()
    .mean()
)

features_to_remove = (
    missing_fraction[
        missing_fraction >= MAX_MISSING_FRACTION
    ]
    .index
    .tolist()
)

print(
    "Frequency features removed due to excessive missingness:"
)

print(features_to_remove)

temporal_frequency_features = [
    c
    for c in temporal_frequency_features
    if c not in features_to_remove
]

print(
    f"\nRemaining Model 4 features: "
    f"{len(temporal_frequency_features)}"
)


# =========================================================
# Dataset subsets
# =========================================================

# ---------------------------------------------------------
# Model 1: Baseline
# ---------------------------------------------------------

normal_baseline = baseline_data[
    baseline_data["experiment_type"] == "train_normal"
].copy()

test_baseline = baseline_data[
    baseline_data["experiment_type"] == "test_anormal"
].copy()


# ---------------------------------------------------------
# Model 2: Raw Window
# ---------------------------------------------------------

normal_raw_window = raw_window_data[
    raw_window_data["experiment_type"] == "train_normal"
].copy()

test_raw_window = raw_window_data[
    raw_window_data["experiment_type"] == "test_anormal"
].copy()


# ---------------------------------------------------------
# Model 3: Temporal
# ---------------------------------------------------------

normal_temporal = temporal_data[
    temporal_data["experiment_type"] == "train_normal"
].copy()

test_temporal = temporal_data[
    temporal_data["experiment_type"] == "test_anormal"
].copy()


# ---------------------------------------------------------
# Model 4: Temporal + Frequency
# ---------------------------------------------------------

normal_temporal_frequency = temporal_frequency_data[
    temporal_frequency_data["experiment_type"] == "train_normal"
].copy()

test_temporal_frequency = temporal_frequency_data[
    temporal_frequency_data["experiment_type"] == "test_anormal"
].copy()


# =========================================================
# Modeling summary
# =========================================================

modeling_summary = pd.DataFrame({
    "model": [
        "Baseline",
        "Raw Window",
        "Temporal",
        "Temporal + Frequency",
    ],

    "dataset": [
        "baseline_raw_windows",
        "window_features",
        "temporal_features",
        "ml_features",
    ],

    "representation": [
        "Raw 60-s signal values",
        "Statistical/process features",
        "Temporal features",
        "Temporal + frequency features",
    ],

    "train_rows": [
        len(normal_baseline),
        len(normal_raw_window),
        len(normal_temporal),
        len(normal_temporal_frequency),
    ],

    "test_rows": [
        len(test_baseline),
        len(test_raw_window),
        len(test_temporal),
        len(test_temporal_frequency),
    ],

    "features": [
        len(baseline_features),
        len(raw_window_features),
        len(temporal_feature_cols),
        len(temporal_frequency_features),
    ],
})


display(modeling_summary)


# =========================================================
# Experiment summary
# =========================================================

group_cols = [
    "experiment_type",
    "experiment",
    "batch",
    "operating_point",
]


dataset_summary = pd.DataFrame({
    "model": [
        "Baseline",
        "Raw Window",
        "Temporal",
        "Temporal + Frequency",
    ],

    "train_experiments": [
        normal_baseline[group_cols]
        .drop_duplicates()
        .shape[0],

        normal_raw_window[group_cols]
        .drop_duplicates()
        .shape[0],

        normal_temporal[group_cols]
        .drop_duplicates()
        .shape[0],

        normal_temporal_frequency[group_cols]
        .drop_duplicates()
        .shape[0],
    ],

    "test_experiments": [
        test_baseline[group_cols]
        .drop_duplicates()
        .shape[0],

        test_raw_window[group_cols]
        .drop_duplicates()
        .shape[0],

        test_temporal[group_cols]
        .drop_duplicates()
        .shape[0],

        test_temporal_frequency[group_cols]
        .drop_duplicates()
        .shape[0],
    ],
})


display(dataset_summary)


# =========================================================
# Generic Isolation Forest function
# =========================================================

def fit_isolation_forest(
    train_df,
    test_df,
    feature_columns,
):
    """
    Fit Isolation Forest on normal training data.

    Missing values are median-imputed using statistics
    learned exclusively from the training data.

    Higher anomaly_score = more anomalous.
    """

    # -----------------------------------------------------
    # Imputation
    # -----------------------------------------------------

    imputer = SimpleImputer(
        strategy="median"
    )

    X_train = imputer.fit_transform(
        train_df[feature_columns]
    )

    X_test = imputer.transform(
        test_df[feature_columns]
    )


    # -----------------------------------------------------
    # Isolation Forest
    # -----------------------------------------------------

    model = IsolationForest(
        n_estimators=300,
        contamination="auto",
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X_train)


    # -----------------------------------------------------
    # Anomaly score
    # -----------------------------------------------------

    result = test_df.copy()

    result["anomaly_score"] = (
        -model.decision_function(X_test)
    )


    # -----------------------------------------------------
    # Internal Isolation Forest prediction
    #
    # Retained for reference only.
    # Final F1 threshold should later be selected
    # on validation data.
    # -----------------------------------------------------

    result["predicted_anomaly"] = (
        model.predict(X_test) == -1
    ).astype(int)


    return model, imputer, result


# =========================================================
# Model 1: Baseline
# =========================================================

model_baseline, imputer_baseline, result_baseline = (
    fit_isolation_forest(
        normal_baseline,
        test_baseline,
        baseline_features,
    )
)


# =========================================================
# Model 2: Raw Window
# =========================================================

model_raw_window, imputer_raw_window, result_raw_window = (
    fit_isolation_forest(
        normal_raw_window,
        test_raw_window,
        raw_window_features,
    )
)


# =========================================================
# Model 3: Temporal
# =========================================================

model_temporal, imputer_temporal, result_temporal = (
    fit_isolation_forest(
        normal_temporal,
        test_temporal,
        temporal_feature_cols,
    )
)


# =========================================================
# Model 4: Temporal + Frequency
# =========================================================

model_temporal_frequency, imputer_temporal_frequency, (
    result_temporal_frequency
) = fit_isolation_forest(
    normal_temporal_frequency,
    test_temporal_frequency,
    temporal_frequency_features,
)


# =========================================================
# Evaluation function
# =========================================================

def evaluate_model(
    model_name,
    df,
):
    """
    Evaluate only sensor-observable anomalies.

    Evaluated labels:

        0 = normal
        2 = observable anomaly
        3 = observable aftereffect

    Label 1 (unobservable fault) is excluded.
    """

    evaluation_mask = (
        df["observable_anomaly"].notna()
        & df["anomaly_score"].notna()
    )

    evaluation_data = df.loc[
        evaluation_mask
    ].copy()

    y_true = (
        evaluation_data["observable_anomaly"]
        .astype(int)
    )

    y_score = (
        evaluation_data["anomaly_score"]
    )

    y_pred = (
        evaluation_data["predicted_anomaly"]
        .astype(int)
    )

    return {
        "Model": model_name,

        "ROC_AUC": roc_auc_score(
            y_true,
            y_score,
        ),

        "Average_Precision": average_precision_score(
            y_true,
            y_score,
        ),

        "Precision": precision_score(
            y_true,
            y_pred,
            zero_division=0,
        ),

        "Recall": recall_score(
            y_true,
            y_pred,
            zero_division=0,
        ),

        "F1": f1_score(
            y_true,
            y_pred,
            zero_division=0,
        ),

        "Accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
    }


# =========================================================
# Compare all four models
# =========================================================

model_comparison = pd.DataFrame([
    evaluate_model(
        "Baseline",
        result_baseline,
    ),

    evaluate_model(
        "Raw Window",
        result_raw_window,
    ),

    evaluate_model(
        "Temporal",
        result_temporal,
    ),

    evaluate_model(
        "Temporal + Frequency",
        result_temporal_frequency,
    ),
])


# =========================================================
# Display comparison
# =========================================================

model_comparison = (
    model_comparison
    .set_index("Model")
)

print("\nModel comparison:")

display(
    model_comparison.round(4)
)


# =========================================================
# Confusion matrices
# =========================================================

for model_name, result in [
    ("Baseline", result_baseline),
    ("Raw Window", result_raw_window),
    ("Temporal", result_temporal),
    ("Temporal + Frequency", result_temporal_frequency),
]:

    evaluation_mask = (
        result["observable_anomaly"].notna()
    )

    y_true = (
        result.loc[
            evaluation_mask,
            "observable_anomaly"
        ].astype(int)
    )

    y_pred = (
        result.loc[
            evaluation_mask,
            "predicted_anomaly"
        ].astype(int)
    )

    print(f"\n{model_name}")
    print("-" * len(model_name))

    display(
        pd.DataFrame(
            confusion_matrix(
                y_true,
                y_pred,
            ),
            index=[
                "actual_normal",
                "actual_anomaly",
            ],
            columns=[
                "pred_normal",
                "pred_anomaly",
            ],
        )
    )


Model 1: 1800 raw signal features
Frequency features removed due to excessive missingness:
['dominant_power_ratio_LS701', 'dominant_power_ratio_LS702']

Remaining Model 4 features: 778


,model,dataset,representation,train_rows,test_rows,features
0,Baseline,baseline_raw_windows,Raw 60-s signal values,3738,7774,1800
1,Raw Window,window_features,Statistical/process features,3738,7774,240
2,Temporal,temporal_features,Temporal features,3738,7774,720
3,Temporal + Frequency,ml_features,Temporal + frequency features,3738,7774,778


,model,train_experiments,test_experiments
0,Baseline,38,81
1,Raw Window,38,81
2,Temporal,38,81
3,Temporal + Frequency,38,81



Model comparison:


,ROC_AUC,Average_Precision,Precision,Recall,F1,Accuracy
Model,,,,,,
Baseline,0.6112,0.4011,0.3663,0.3957,0.3804,0.6094
Raw Window,0.7267,0.5061,0.5110,0.4962,0.5035,0.7034
Temporal,0.7630,0.5683,0.5281,0.3774,0.4402,0.7091
Temporal + Frequency,0.7637,0.5801,0.5407,0.3796,0.4461,0.7143


...

,pred_normal,pred_anomaly
actual_normal,4086,1063
actual_anomaly,1128,1111



Temporal
--------


,pred_normal,pred_anomaly
actual_normal,4394,755
actual_anomaly,1394,845



Temporal + Frequency
--------------------


,pred_normal,pred_anomaly
actual_normal,4427,722
actual_anomaly,1389,850


In [2]:
# =========================================================
# Create locked validation / final-test experiment split
# =========================================================

from pathlib import Path

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit


# =========================================================
# 1. Load baseline raw windows
# =========================================================

BASELINE_RAW_WINDOWS_PATH = (
    "../data/processed/baseline_raw_windows.parquet"
)

baseline_raw_windows = pd.read_parquet(
    BASELINE_RAW_WINDOWS_PATH
)


# =========================================================
# 2. Create experiment-level summary
# =========================================================

experiment_summary = (
    baseline_raw_windows
    .groupby("identifier")
    .agg(
        batch=("batch", "first"),
        operating_point=("operating_point", "first"),
        experiment_type=("experiment_type", "first"),
        experiment=("experiment", "first"),
        anomaly_labels=(
            "anomaly_label",
            lambda x: set(x.dropna())
        ),
    )
    .reset_index()
)


# =========================================================
# 3. Classify experiments
# =========================================================

experiment_summary["experiment_class"] = "unknown"

experiment_summary.loc[
    experiment_summary["experiment_type"] == "train_normal",
    "experiment_class"
] = "normal"

experiment_summary.loc[
    experiment_summary["anomaly_labels"].apply(
        lambda x: bool(x.intersection({2, 3}))
    ),
    "experiment_class"
] = "observable_anomaly"

experiment_summary.loc[
    experiment_summary["anomaly_labels"].apply(
        lambda x: 1 in x and not bool(x.intersection({2, 3}))
    ),
    "experiment_class"
] = "unobservable_fault"


# =========================================================
# 4. Select observable anomaly experiments
# =========================================================

observable_experiments = experiment_summary[
    experiment_summary["experiment_class"]
    == "observable_anomaly"
].copy()


# =========================================================
# 5. Create batch + operating-point groups
# =========================================================

groups = (
    observable_experiments[
        ["batch", "operating_point"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

group_sizes = (
    observable_experiments
    .groupby(
        ["batch", "operating_point"]
    )
    .size()
    .reset_index(name="n_experiments")
)

groups = groups.merge(
    group_sizes,
    on=["batch", "operating_point"],
    how="left"
)


# =========================================================
# 6. Find balanced validation / final-test split
# =========================================================

target_experiments = (
    len(observable_experiments) / 2
)

best_difference = float("inf")
best_split = None

for random_state in range(1000):

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.5,
        random_state=random_state
    )

    validation_idx, final_test_idx = next(
        splitter.split(
            groups,
            groups=groups.index
        )
    )

    validation_candidate = groups.iloc[
        validation_idx
    ]

    validation_n = (
        validation_candidate["n_experiments"]
        .sum()
    )

    difference = abs(
        validation_n - target_experiments
    )

    if difference < best_difference:

        best_difference = difference

        best_split = (
            validation_candidate.copy(),
            groups.iloc[final_test_idx].copy(),
            random_state
        )


# =========================================================
# 7. Assign split labels
# =========================================================

validation_groups = best_split[0]
final_test_groups = best_split[1]

validation_groups["split"] = "validation"
final_test_groups["split"] = "final_test"

split_groups = pd.concat(
    [
        validation_groups,
        final_test_groups
    ],
    ignore_index=True
)


# =========================================================
# 8. Apply split to experiments
# =========================================================

observable_experiments = (
    observable_experiments
    .merge(
        split_groups[
            [
                "batch",
                "operating_point",
                "split"
            ]
        ],
        on=[
            "batch",
            "operating_point"
        ],
        how="left",
        validate="many_to_one"
    )
)


# =========================================================
# 9. Verify and save
# =========================================================

# Validate experiment-level split assignment
assert observable_experiments["split"].notna().all()

experiment_split_counts = (
    observable_experiments["split"]
    .value_counts()
    .to_dict()
)

assert experiment_split_counts == {
    "validation": 38,
    "final_test": 39,
}

# Validate group-level split assignment
group_split_counts = (
    observable_experiments[
        ["batch", "operating_point", "split"]
    ]
    .drop_duplicates()
    ["split"]
    .value_counts()
    .to_dict()
)

assert group_split_counts == {
    "validation": 20,
    "final_test": 20,
}

# Validate that no (batch, operating_point) group is split
# between validation and final test
group_split_check = (
    observable_experiments
    .groupby(["batch", "operating_point"])["split"]
    .nunique()
)

assert group_split_check.max() == 1

assert (
    set(
        zip(
            validation_groups["batch"],
            validation_groups["operating_point"]
        )
    )
    .isdisjoint(
        set(
            zip(
                final_test_groups["batch"],
                final_test_groups["operating_point"]
            )
        )
    )
)


split_to_save = observable_experiments[
    [
        "identifier",
        "batch",
        "operating_point",
        "experiment",
        "experiment_type",
        "experiment_class",
        "split"
    ]
].copy()


OUTPUT_PATH = Path(
    "../data/processed/experiment_split.parquet"
)

split_to_save.to_parquet(
    OUTPUT_PATH,
    index=False
)

print(
    f"Locked split saved to: {OUTPUT_PATH}"
)

Locked split saved to: ..\data\processed\experiment_split.parquet
